In [46]:
from utils.constants import *
import shutil
import kagglehub
"""
pip install kagglehub[pandas-datasets] to download dataset
"""

os.makedirs('gtrsb_dataset', exist_ok=True)
dataset_path = kagglehub.dataset_download("meowmeowmeowmeowmeow/gtsrb-german-traffic-sign")
print(f"Dataset downloaded to: {dataset_path}")

# move the dataset to current project directory
target_path = './gtrsb_dataset'
if not os.path.exists(target_path):
    shutil.move(dataset_path, target_path)

Dataset downloaded to: /home/eddie/.cache/kagglehub/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign/versions/1


In [47]:
from pathlib import Path
import pandas as pd

train_df = pd.read_csv(os.path.join(DF_PATH, 'Train.csv'))
test_df = pd.read_csv(os.path.join(DF_PATH, 'Test.csv'))
meta_df = pd.read_csv(os.path.join(DF_PATH, 'Meta.csv'))
for df in [train_df, test_df]:
      df["roi_w"] = (df["Roi.X2"] - df["Roi.X1"]).clip(lower=1)
      df["roi_h"] = (df["Roi.Y2"] - df["Roi.Y1"]).clip(lower=1)
      df["img_area"] = df["Width"] * df["Height"]
      df["roi_area"] = df["roi_w"] * df["roi_h"]
      df["roi_cover"] = (df["roi_area"] / df["img_area"]).clip(upper=1.0)

train_df.head(3)


,Width,Height,Roi.X1,Roi.Y1,Roi.X2,Roi.Y2,ClassId,Path,roi_w,roi_h,img_area,roi_area,roi_cover
0,27,26,5,5,22,20,20,Train/20/00020_00000_00000.png,17,15,702,255,0.363248
1,28,27,5,6,23,22,20,Train/20/00020_00000_00001.png,18,16,756,288,0.380952
2,29,26,6,5,24,21,20,Train/20/00020_00000_00002.png,18,16,754,288,0.381963


In [48]:

class_column = meta_df['ClassId'].to_list()

test_df = test_df.merge(
    meta_df[['ClassId', 'ColorId', 'ShapeId', 'SignId']],
    how='left',      # keep all rows in test_df
    on='ClassId'     # match by ClassId
)
train_df = train_df.merge(
    meta_df[['ClassId', 'ColorId', 'ShapeId', 'SignId']],
    how='left',      # keep all rows in test_df
    on='ClassId'     # match by ClassId
)


#test_df.rename(columns={'ColorId': 'Color'}, inplace=True)
train_df

,Width,Height,Roi.X1,Roi.Y1,Roi.X2,Roi.Y2,ClassId,Path,roi_w,roi_h,img_area,roi_area,roi_cover,ColorId,ShapeId,SignId
0,27,26,5,5,22,20,20,Train/20/00020_00000_00000.png,17,15,702,255,0.363248,0,0,1.1
1,28,27,5,6,23,22,20,Train/20/00020_00000_00001.png,18,16,756,288,0.380952,0,0,1.1
2,29,26,6,5,24,21,20,Train/20/00020_00000_00002.png,18,16,754,288,0.381963,0,0,1.1
3,28,27,5,6,23,22,20,Train/20/00020_00000_00003.png,18,16,756,288,0.380952,0,0,1.1
4,28,26,5,5,23,21,20,Train/20/00020_00000_00004.png,18,16,728,288,0.395604,0,0,1.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39204,52,56,5,6,47,51,42,Train/42/00042_00007_00025.png,42,45,2912,1890,0.649038,3,1,3.28
39205,56,58,5,5,51,53,42,Train/42/00042_00007_00026.png,46,48,3248,2208,0.679803,3,1,3.28
39206,58,62,5,6,53,57,42,Train/42/00042_00007_00027.png,48,51,3596,2448,0.680756,3,1,3.28
39207,63,69,5,7,58,63,42,Train/42/00042_00007_00028.png,53,56,4347,2968,0.682770,3,1,3.28


In [49]:
# Combine train and test so you can perform manual splits / KFold later
train_df = train_df.copy()
test_df = test_df.copy()

# train_df['_source'] = 'train'
# test_df['_source'] = 'test'

combined_df = pd.concat([train_df, test_df], ignore_index=True)

# save combined dataset for later use
combined_path = os.path.join(DF_PATH, 'Combined.csv')
combined_df.to_csv(combined_path, index=False)

print(f"Combined shape: {combined_df.shape} -> saved to {combined_path}")
combined_df.head()

Combined shape: (51839, 16) -> saved to gtrsb_dataset/1/Combined.csv


,Width,Height,Roi.X1,Roi.Y1,Roi.X2,Roi.Y2,ClassId,Path,roi_w,roi_h,img_area,roi_area,roi_cover,ColorId,ShapeId,SignId
0,27,26,5,5,22,20,20,Train/20/00020_00000_00000.png,17,15,702,255,0.363248,0,0,1.1
1,28,27,5,6,23,22,20,Train/20/00020_00000_00001.png,18,16,756,288,0.380952,0,0,1.1
2,29,26,6,5,24,21,20,Train/20/00020_00000_00002.png,18,16,754,288,0.381963,0,0,1.1
3,28,27,5,6,23,22,20,Train/20/00020_00000_00003.png,18,16,756,288,0.380952,0,0,1.1
4,28,26,5,5,23,21,20,Train/20/00020_00000_00004.png,18,16,728,288,0.395604,0,0,1.1


In [50]:
from sklearn.model_selection import train_test_split

# stratify by ClassId if present, otherwise no stratify
label_col = 'ClassId' if 'ClassId' in combined_df.columns else None
stratify_vals = combined_df[label_col] if label_col is not None else None

# select only columns needed for training
keep_cols = ['ClassId', 'ColorId', 'ShapeId', 'SignId']
clean_df = combined_df[[c for c in keep_cols if c in combined_df.columns]].copy()

train_manual, test_manual = train_test_split(
    clean_df,
    test_size=0.2,
    random_state=42,
    stratify=stratify_vals
)


train_out = os.path.join(DF_PATH, 'Train_manual.csv')
test_out = os.path.join(DF_PATH, 'Test_manual.csv')
train_manual.to_csv(train_out, index=False)
test_manual.to_csv(test_out, index=False)

print(f"Saved Train_manual ({train_manual.shape}) -> {train_out}")
print(f"Saved Test_manual  ({test_manual.shape}) -> {test_out}")

if label_col:
    print("Train class distribution (normalized):")
    print(train_manual[label_col].value_counts(normalize=True).sort_index())
    print("Test class distribution (normalized):")
    print(test_manual[label_col].value_counts(normalize=True).sort_index())


Saved Train_manual ((41471, 4)) -> gtrsb_dataset/1/Train_manual.csv
Saved Test_manual  ((10368, 4)) -> gtrsb_dataset/1/Test_manual.csv
Train class distribution (normalized):
ClassId
0     0.005208
1     0.056714
2     0.057872
3     0.035880
4     0.050927
5     0.048034
6     0.010996
7     0.036459
8     0.035880
9     0.037617
10    0.051506
11    0.033566
12    0.053821
13    0.055557
14    0.020255
15    0.016204
16    0.010996
17    0.028357
18    0.030672
19    0.005208
20    0.008681
21    0.008102
22    0.009838
23    0.012732
24    0.006945
25    0.038195
26    0.015047
27    0.005787
28    0.013311
29    0.006945
30    0.011574
31    0.020255
32    0.005787
33    0.017337
34    0.010417
35    0.030672
36    0.009838
37    0.005208
38    0.053242
39    0.007523
40    0.008681
41    0.005787
42    0.006366
Name: proportion, dtype: float64
Test class distribution (normalized):
ClassId
0     0.005208
1     0.056713
2     0.057870
3     0.035880
4     0.050926
5     0.048032
6   

In [51]:
from sklearn.calibration import LabelEncoder

# Encode SignId
le = LabelEncoder()
train_manual['SignId'] = le.fit_transform(train_manual['SignId'])
test_manual['SignId'] = le.transform(test_manual['SignId'])

train_manual.head()

,ClassId,ColorId,ShapeId,SignId
14932,9,0,1,18
12635,8,0,1,22
5947,3,0,1,22
15718,10,0,1,20
17499,11,0,0,3


In [52]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X = train_manual[['ColorId', 'ShapeId']] 
y = train_manual['ClassId'] 


model = KNeighborsClassifier(n_neighbors= 5)

accuracies = []

# Define the hyperparameter grid
param_grid = {
    'n_neighbors': [7, 9, 11, 15, 20],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]  # 1=Manhattan, 2=Euclidean
}

# Use StratifiedKFold for cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=skf,
    scoring='accuracy',
    n_jobs=-1  # use all cores
)

grid.fit(X, y)

print("Best hyperparameters:", grid.best_params_)
print("Best cross-validation accuracy:", grid.best_score_)

Best hyperparameters: {'n_neighbors': 15, 'p': 1, 'weights': 'uniform'}
Best cross-validation accuracy: 0.2504400095757581


In [54]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Features and labels
X = train_manual[['ColorId', 'ShapeId']]  # your features
y = train_manual['ClassId']              # labels

# Step 1: Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # fit on training data

# Step 2: Define SVM model
model = SVC()

# Step 3: Hyperparameter grid
param_grid = {
    'C': [0.1, 1, 10, 50],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto']
}

# Step 4: Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Step 5: Grid Search
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=skf,
    scoring='accuracy',
    n_jobs=-1
)

# Step 6: Fit the model
grid.fit(X_scaled, y)

print("Best hyperparameters:", grid.best_params_)
print("Best cross-validation accuracy:", grid.best_score_)


Best hyperparameters: {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
Best cross-validation accuracy: 0.2899375497193044


- Split dataset
- knn trainieren
- hyperparameter